In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

In [ ]:
pairs = [
    ("eu sunt student",      "i am a student"),
    ("tu esti student",      "you are a student"),
    ("el este student",      "he is a student"),
    ("ea este studenta",     "she is a student"),
    ("eu sunt profesor",     "i am a teacher"),
    ("tu esti profesor",     "you are a teacher"),
    ("el este profesor",     "he is a teacher"),
    ("ea este profesoara",   "she is a teacher"),
    ("eu citesc o carte",    "i read a book"),
    ("tu citesti o carte",   "you read a book"),
    ("el citeste o carte",   "he reads a book"),
    ("ea citeste o carte",   "she reads a book"),
    ("eu am o pisica",       "i have a cat"),
    ("tu ai o pisica",       "you have a cat"),
    ("el are o pisica",      "he has a cat"),
    ("ea are o pisica",      "she has a cat"),
    ("eu am un caine",       "i have a dog"),
    ("tu ai un caine",       "you have a dog"),
    ("el are un caine",      "he has a dog"),
    ("ea are un caine",      "she has a dog"),
    ("ana are mere",         "ana has apples"),
    ("ana citeste o carte",  "ana reads a book"),
    ("eu iubesc cafeaua",    "i love coffee"),
    ("el iubeste cafeaua",   "he loves coffee"),
    ("noi suntem studenti",  "we are students"),
    ("noi avem o pisica",    "we have a cat"),
]

In [ ]:
PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIAL_TOKENS = ["<pad>", "<start>", "<end>", "<unk>"]

def build_vocab(sentences):
  words = sorted({w for s in sentences for w in s.split()})
  itos = SPECIAL_TOKENS + words
  stoi = {w : i for i, w in enumerate(itos)}

  return stoi, itos

src_stoi, src_itos = build_vocab([ro for ro, en in pairs])
tgt_stoi, tgt_itos = build_vocab([en for ro, en in pairs])


In [ ]:
src_stoi

{'<pad>': 0,
 '<start>': 1,
 '<end>': 2,
 '<unk>': 3,
 'ai': 4,
 'am': 5,
 'ana': 6,
 'are': 7,
 'avem': 8,
 'cafeaua': 9,
 'caine': 10,
 'carte': 11,
 'citesc': 12,
 'citeste': 13,
 'citesti': 14,
 'ea': 15,
 'el': 16,
 'este': 17,
 'esti': 18,
 'eu': 19,
 'iubesc': 20,
 'iubeste': 21,
 'mere': 22,
 'noi': 23,
 'o': 24,
 'pisica': 25,
 'profesoara': 26,
 'profesor': 27,
 'student': 28,
 'studenta': 29,
 'studenti': 30,
 'sunt': 31,
 'suntem': 32,
 'tu': 33,
 'un': 34}

In [ ]:
src_itos

['<pad>',
 '<start>',
 '<end>',
 '<unk>',
 'ai',
 'am',
 'ana',
 'are',
 'avem',
 'cafeaua',
 'caine',
 'carte',
 'citesc',
 'citeste',
 'citesti',
 'ea',
 'el',
 'este',
 'esti',
 'eu',
 'iubesc',
 'iubeste',
 'mere',
 'noi',
 'o',
 'pisica',
 'profesoara',
 'profesor',
 'student',
 'studenta',
 'studenti',
 'sunt',
 'suntem',
 'tu',
 'un']

In [ ]:
def encode(sentence, stoi):
  return [stoi.get(word, UNK) for word in sentence.split()]

src_list = [encode(ro, src_stoi) for ro, en in pairs]
tgt_list = [[BOS] + encode(en, tgt_stoi) + [EOS] for ro, en in pairs]

def pad_batch(sequences):
  max_len = max(len(s) for s in sequences)
  return torch.tensor([s + [PAD] * (max_len - len(s)) for s in sequences])

src = pad_batch(src_list)
tgt = pad_batch(tgt_list)


In [ ]:
src.shape

torch.Size([26, 4])

In [ ]:
src[20]

tensor([ 6,  7, 22,  0])

In [ ]:
tgt.shape

torch.Size([26, 6])

In [ ]:
tgt[20]

tensor([ 1,  6, 13,  7,  2,  0])

In [ ]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=100):
      super().__init__()
      pe = torch.zeros(max_len, d_model)
      position = torch.arange(0, max_len).unsqueeze(1).float()

      div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))

      pe[:, 0::2] = torch.sin(position * div_term)
      pe[:, 1::2] = torch.cos(position * div_term)

      self.register_buffer("pe", pe.unsqueeze(0))

  def forward(self, x):
    ## Adunam codificarea pozitionala la embedding-uri
    return x + self.pe[:, :x.size(1)]




In [ ]:
pos_demo = PositionalEncoding(d_model=8)
pos_demo.pe[0, 1].round(decimals=2).tolist()

[0.8399999737739563,
 0.5400000214576721,
 0.10000000149011612,
 1.0,
 0.009999999776482582,
 1.0,
 0.0,
 1.0]

In [ ]:
class Translator(nn.Module):
  def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, nhead=4, num_layers=2, dim_feed_forward=128):
    super().__init__()
    self.d_model = d_model

    self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD)
    self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD)
    self.pos_encoding = PositionalEncoding(d_model)

    self.transformer = nn.Transformer(
        d_model = d_model,
        nhead=nhead,
        num_encoder_layers=num_layers,
        num_decoder_layers=num_layers,
        dim_feedforward=dim_feed_forward,
        dropout=0.1,
        batch_first=True
    )

    self.transformer.encoder.use_nested_tensor = False
    self.output_layer = nn.Linear(d_model, tgt_vocab_size)


  def forward(self, src, tgt_input):
    src_emb = self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
    tgt_emb = self.pos_encoding(self.tgt_embedding(tgt_input) * math.sqrt(self.d_model))

    seq_len = tgt_input.size(1)
    causal_mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=tgt_input.device), diagonal=1
    )

    output = self.transformer(
        src_emb, tgt_emb,
        tgt_mask = causal_mask,
        src_key_padding_mask=(src==PAD),
        tgt_key_padding_mask=(tgt_input==PAD),
        memory_key_padding_mask=(src == PAD)
    )

    return self.output_layer(output)





In [ ]:
model = Translator(len(src_stoi), len(tgt_stoi))


In [ ]:
sum(p.numel() for p in model.parameters())

173532

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=400)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
for epoch in range(1, 301):
  tgt_input = tgt[:, :-1]
  tgt_output = tgt[:, 1:]

  logits = model(src, tgt_input)

  loss = criterion(
      logits.reshape(-1, logits.size(-1)), # batch * lungime, marime_vocabular
      tgt_output.reshape(-1)
  )

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 50 == 0:
    print(f"Epoch: {epoch} | loss {loss.item():.4f}")



Epoch: 50 | loss 0.3926
Epoch: 100 | loss 0.0734
Epoch: 150 | loss 0.0353
Epoch: 200 | loss 0.0219
Epoch: 250 | loss 0.0126
Epoch: 300 | loss 0.0098


In [ ]:
@torch.no_grad()
def translate(sentence, max_len=12):
  model.eval()
  src_tensor = torch.tensor([encode(sentence, src_stoi)])
  generated = torch.tensor([[BOS]]) #Pornim doar cu <start>

  for _ in range(max_len):
    logits = model(src_tensor, generated)
    next_token = logits[0, -1].argmax().item()
    if next_token == EOS:
      break
    generated = torch.cat([generated, torch.tensor([[next_token]])], dim=1)


  return " ".join(tgt_itos[i] for i in generated[0, 1:].tolist())



In [ ]:
corect = 0
for ro, en in pairs:
  prediction = translate(ro)
  is_corect = (prediction == en)
  corect += is_corect
  status = "OK " if is_corect else "GRESIT"
  print(f"[{status}] {ro} -> {prediction}")

[OK ] eu sunt student -> i am a student
[OK ] tu esti student -> you are a student
[OK ] el este student -> he is a student
[OK ] ea este studenta -> she is a student
[OK ] eu sunt profesor -> i am a teacher
[OK ] tu esti profesor -> you are a teacher
[OK ] el este profesor -> he is a teacher
[OK ] ea este profesoara -> she is a teacher
[OK ] eu citesc o carte -> i read a book
[OK ] tu citesti o carte -> you read a book
[OK ] el citeste o carte -> he reads a book
[OK ] ea citeste o carte -> she reads a book
[OK ] eu am o pisica -> i have a cat
[OK ] tu ai o pisica -> you have a cat
[OK ] el are o pisica -> he has a cat
[OK ] ea are o pisica -> she has a cat
[OK ] eu am un caine -> i have a dog
[OK ] tu ai un caine -> you have a dog
[OK ] el are un caine -> he has a dog
[OK ] ea are un caine -> she has a dog
[OK ] ana are mere -> ana has apples
[OK ] ana citeste o carte -> ana reads a book
[OK ] eu iubesc cafeaua -> i love coffee
[OK ] el iubeste cafeaua -> he loves coffee
[OK ] noi sun

In [ ]:
for sentence in ["ea are mere", "el citeste o carte", "tu esti o carte", "ana are o pisica"]:
  print(f"{sentence} -> {translate(sentence)}")

ea are mere -> she has a dog
el citeste o carte -> he reads a book
tu esti o carte -> you read a book
ana are o pisica -> ana has a cat
